In [1]:
print("hello")

hello


In [2]:
import pandas as pd
df = pd.DataFrame()

In [3]:
import json
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

In [4]:
json_file_path = "1000.json"

In [5]:
with open(json_file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

In [6]:
print(type(data))
print(len(data))
print(data[0])

<class 'list'>
1000
{'customer_id': 1, 'customer_name': 'Rafi Khan', 'email': 'rafi.khan1@example.com', 'amount': 2598.57, 'transaction_date': '2024-10-08'}


In [7]:
df = pd.DataFrame(data)

In [8]:
print(df.head())

   customer_id customer_name                      email    amount  \
0            1     Rafi Khan     rafi.khan1@example.com   2598.57   
1            2  Jamal Rahman  jamal.rahman2@example.com  14039.84   
2            3   Rafi Sarker   rafi.sarker3@example.com   8785.19   
3            4   Rahim Ahmed   rahim.ahmed4@example.com   9460.15   
4            5  Ayesha Hasan  ayesha.hasan5@example.com   2750.94   

  transaction_date  
0       2024-10-08  
1       2024-04-14  
2       2025-03-08  
3       2024-08-26  
4       2024-07-22  


In [9]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       1000 non-null   int64 
 1   customer_name     993 non-null    str   
 2   email             1000 non-null   str   
 3   amount            1000 non-null   object
 4   transaction_date  1000 non-null   str   
dtypes: int64(1), object(1), str(3)
memory usage: 86.3+ KB
None


In [10]:
df["customer_name"] = df["customer_name"].str.strip()
df["customer_name"] = df["customer_name"].fillna("Unknown")

In [11]:
df["email"] = df["email"].str.lower()
df["email"] = df["email"].replace("", None)

In [12]:
df["amount"] = (
    df["amount"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)

In [13]:
df["amount"] = df["amount"].fillna(0)

In [14]:
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    dayfirst=True,
    errors="coerce"
)
df["transaction_date"] = df["transaction_date"].dt.date

In [15]:
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())
print(df["customer_id"].duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       1000 non-null   int64  
 1   customer_name     1000 non-null   str    
 2   email             992 non-null    str    
 3   amount            1000 non-null   float64
 4   transaction_date  406 non-null    object 
dtypes: float64(1), int64(1), object(1), str(2)
memory usage: 76.5+ KB
None
customer_id           0
customer_name         0
email                 8
amount                0
transaction_date    594
dtype: int64
0
0


In [ ]:
connection = psycopg2.connect(
    host="localhost",
    port="5432",
    database="Database Name",
    user="postgres",
    password="********"
)

In [17]:
cursor = connection.cursor()
create_table_query = """
CREATE TABLE IF NOT EXISTS customer_transactions (
    customer_id INTEGER PRIMARY KEY,
    customer_name VARCHAR(100),
    email VARCHAR(150),
    amount NUMERIC(12, 2),
    transaction_date DATE
);
"""

In [18]:
cursor.execute(create_table_query)
connection.commit()

In [19]:
insert_query = """
INSERT INTO customer_transactions (
    customer_id,
    customer_name,
    email,
    amount,
    transaction_date
)
VALUES (%s, %s, %s, %s, %s)
ON CONFLICT (customer_id) DO NOTHING;
"""
records = df[
    [
        "customer_id",
        "customer_name",
        "email",
        "amount",
        "transaction_date"
    ]
].values.tolist()

In [20]:
connection.rollback()

In [21]:
print(type(records))
print(len(records))
print(records[:3])
records = [
    [
        customer_id,
        customer_name,
        email,
        amount,
        None if pd.isna(transaction_date) else transaction_date,
    ]
    for customer_id, customer_name, email, amount, transaction_date in records
]

execute_batch(
    cursor,
    insert_query,
    records,
    page_size=100
)

connection.commit()

print(f"{len(records)} records inserted successfully!")

connection.commit()

print(f"{len(records)} records inserted successfully!")

<class 'list'>
1000
[[1, 'Rafi Khan', 'rafi.khan1@example.com', 2598.57, datetime.date(2024, 8, 10)], [2, 'Jamal Rahman', 'jamal.rahman2@example.com', 14039.84, NaT], [3, 'Rafi Sarker', 'rafi.sarker3@example.com', 8785.19, datetime.date(2025, 8, 3)]]


1000 records inserted successfully!
1000 records inserted successfully!


In [22]:
print(df.dtypes)
print(df.head(10))
print(df["customer_id"].isnull().sum())
print(df["customer_id"].duplicated().sum())

customer_id           int64
customer_name           str
email                   str
amount              float64
transaction_date     object
dtype: object
   customer_id   customer_name                        email    amount  \
0            1       Rafi Khan       rafi.khan1@example.com   2598.57   
1            2    Jamal Rahman    jamal.rahman2@example.com  14039.84   
2            3     Rafi Sarker     rafi.sarker3@example.com   8785.19   
3            4     Rahim Ahmed     rahim.ahmed4@example.com   9460.15   
4            5    Ayesha Hasan    ayesha.hasan5@example.com   2750.94   
5            6  Sumaiya Sarker  sumaiya.sarker6@example.com  42010.03   
6            7     Fahim Islam     fahim.islam7@example.com  80962.10   
7            8  Tanvir Hossain  tanvir.hossain8@example.com  69844.13   
8            9   Sakib Hossain   sakib.hossain9@example.com  21609.84   
9           10      Karim Khan     karim.khan10@example.com  38054.74   

  transaction_date  
0       2024-08-10  


In [23]:
print(records[:5])

[[1, 'Rafi Khan', 'rafi.khan1@example.com', 2598.57, datetime.date(2024, 8, 10)], [2, 'Jamal Rahman', 'jamal.rahman2@example.com', 14039.84, None], [3, 'Rafi Sarker', 'rafi.sarker3@example.com', 8785.19, datetime.date(2025, 8, 3)], [4, 'Rahim Ahmed', 'rahim.ahmed4@example.com', 9460.15, None], [5, 'Ayesha Hasan', 'ayesha.hasan5@example.com', 2750.94, None]]


In [24]:
cursor.execute(insert_query, records[0])
connection.commit()

In [25]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

print("Everything OK")

Everything OK
